# Test avec CatBoostRegressor

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

In [2]:
df = pd.read_csv("../data/gold.csv")

In [3]:
# column_titles = df.columns.tolist()
# column_titles

In [4]:
X = df.drop(["prix_median"], axis=1)
y = df['prix_median']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, shuffle=True, random_state=42)

categorial_features = ["proximité_autoroute"]

numerical_features = ['tx_crim', 'tx_residence','tx_commerce',
       'tx_nitriq', 'nb_piece', 'tx_ancienneté_parc_immo', 'distance_centre_emploi',
        'indice_impot_foncier',  'ratio_eleve_enseignant', 'tx_status_sociaux_eco_inf']

categorical_transformer = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
numerical_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorial_features),
        ('num', numerical_transformer, numerical_features)
    ],
    remainder="passthrough" 
)

# Créer un objet CatBoostRegressor
# model = CatBoostRegressor(random_state=42, verbose=False)
# model = CatBoostRegressor(depth=4, learning_rate=0.19293408926881367, l2_leaf_reg=1.9445149121337306, random_state=42, verbose=False)
# model = CatBoostRegressor(iterations = 700, learning_rate = 0.0693060439562983, depth = 6, loss_function = 'RMSE', subsample = 0.30000000000000004, l2_leaf_reg = 0.9500743944149637, random_strength = 0.10292853111379426, random_state=42, verbose=False)
model = CatBoostRegressor(iterations = 900, learning_rate = 0.0677148890414124, depth = 9, loss_function = 'RMSE', subsample = 0.5, l2_leaf_reg = 9.760330771250379, random_strength = 4.695597681100087, random_state=42, verbose=False)


# Créer un pipeline avec le préprocesseur et le modèle CatBoostRegressor
pipe = Pipeline([
     ('preprocessor', preprocessor),
     ('model', model)
])

# Entraîner le pipeline sur les données d'entraînement
pipe.fit(X_train, y_train)
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

print("Mean squared error : ")
print("TRAIN :",mean_squared_error(y_train, y_pred_train))
print("TEST :",mean_squared_error(y_test, y_pred_test))

# print("/////////////////////////////////////////////////////")

print("Mean absolute error : ")
print("TRAIN :",mean_absolute_error(y_train, y_pred_train))
print("TEST :",mean_absolute_error(y_test, y_pred_test))

# print("/////////////////////////////////////////////////////")

print("R2 score : ")    
print("TRAIN :",r2_score(y_train, y_pred_train))
print("TEST :",r2_score(y_test, y_pred_test))

Mean squared error : 
TRAIN : 0.1559923129043852
TEST : 8.50384599912411
Mean absolute error : 
TRAIN : 0.31679518729962375
TEST : 1.9309213145027615
R2 score : 
TRAIN : 0.9982043720399821
TEST : 0.8840391749525192


### Optuna (recherche des hyperparametres)

In [5]:
import optuna

In [6]:
def objective(trial):
    # Définition des hyperparamètres à optimiser
    iterations = trial.suggest_int('iterations', 100, 1000, step=100)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.1, log=True)
    depth = trial.suggest_int('depth', 3, 10)
    loss_function = trial.suggest_categorical('loss_function', ['RMSE', 'MAE'])
    subsample = trial.suggest_float('subsample', 0.1, 1.0, step=0.1)
    l2_leaf_reg = trial.suggest_float('l2_leaf_reg', 0.1, 10.0)
    random_strength = trial.suggest_float('random_strength', 0.1, 10.0)
    
    # Création du modèle avec les hyperparamètres suggérés
    model = CatBoostRegressor(iterations=iterations,
                              learning_rate=learning_rate,
                              depth=depth,
                              loss_function=loss_function,
                              subsample=subsample,
                              l2_leaf_reg=l2_leaf_reg,
                              random_strength=random_strength,
                              random_state=42,
                              verbose=False,
                              allow_writing_files=False)
    
    # Même pipeline (préprocesseur + modèle) que pour l'évaluation finale
    pipe = Pipeline([
         ('preprocessor', preprocessor),
         ('model', model)
    ])
    
    # Validation croisée sur le jeu d'entraînement uniquement :
    # le jeu de test reste réservé à l'évaluation finale
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    r2 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1).mean()
    
    return r2

In [7]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-10-08 11:58:52,274] A new study created in memory with name: no-name-20ef7cc1-e97d-411f-bf5f-de97c4009d4b


[I 2026-10-08 11:58:55,097] Trial 0 finished with value: 0.7963077889994393 and parameters: {'iterations': 500, 'learning_rate': 0.009342006121782364, 'depth': 7, 'loss_function': 'MAE', 'subsample': 0.6, 'l2_leaf_reg': 5.8160683574466265, 'random_strength': 7.497503264839211}. Best is trial 0 with value: 0.7963077889994393.


[I 2026-10-08 11:59:02,941] Trial 1 finished with value: 0.7614521468292887 and parameters: {'iterations': 800, 'learning_rate': 0.002673501373839284, 'depth': 9, 'loss_function': 'RMSE', 'subsample': 1.0, 'l2_leaf_reg': 0.5605495089344696, 'random_strength': 4.444786684133386}. Best is trial 0 with value: 0.7963077889994393.


[I 2026-10-08 11:59:04,499] Trial 2 finished with value: 0.8326462105470173 and parameters: {'iterations': 1000, 'learning_rate': 0.009518892515597465, 'depth': 5, 'loss_function': 'RMSE', 'subsample': 0.2, 'l2_leaf_reg': 8.792986521058275, 'random_strength': 8.251860573169864}. Best is trial 2 with value: 0.8326462105470173.


[I 2026-10-08 11:59:07,819] Trial 3 finished with value: 0.608409211910077 and parameters: {'iterations': 1000, 'learning_rate': 0.0013961912425692437, 'depth': 7, 'loss_function': 'RMSE', 'subsample': 1.0, 'l2_leaf_reg': 5.914175409847862, 'random_strength': 1.3332791535596162}. Best is trial 2 with value: 0.8326462105470173.


...

[I 2026-10-08 12:00:28,473] Trial 95 finished with value: 0.8848517060636109 and parameters: {'iterations': 900, 'learning_rate': 0.023919888976284682, 'depth': 5, 'loss_function': 'RMSE', 'subsample': 1.0, 'l2_leaf_reg': 1.9833655646985342, 'random_strength': 1.6589616616355984}. Best is trial 88 with value: 0.8872374115294746.


[I 2026-10-08 12:00:29,599] Trial 96 finished with value: 0.8819522539281601 and parameters: {'iterations': 800, 'learning_rate': 0.021088532391860367, 'depth': 5, 'loss_function': 'RMSE', 'subsample': 1.0, 'l2_leaf_reg': 1.5601591739062153, 'random_strength': 0.30183010605798244}. Best is trial 88 with value: 0.8872374115294746.


[I 2026-10-08 12:00:30,798] Trial 97 finished with value: 0.882102881465866 and parameters: {'iterations': 900, 'learning_rate': 0.031133345561211796, 'depth': 5, 'loss_function': 'RMSE', 'subsample': 1.0, 'l2_leaf_reg': 2.1743958978506215, 'random_strength': 1.5680038885416807}. Best is trial 88 with value: 0.8872374115294746.


[I 2026-10-08 12:00:31,867] Trial 98 finished with value: 0.885684941525203 and parameters: {'iterations': 900, 'learning_rate': 0.035868000680184184, 'depth': 5, 'loss_function': 'RMSE', 'subsample': 1.0, 'l2_leaf_reg': 2.9672228847457456, 'random_strength': 0.42658762364844993}. Best is trial 88 with value: 0.8872374115294746.


[I 2026-10-08 12:00:33,210] Trial 99 finished with value: 0.8834909398249394 and parameters: {'iterations': 900, 'learning_rate': 0.019516809974785738, 'depth': 5, 'loss_function': 'RMSE', 'subsample': 1.0, 'l2_leaf_reg': 1.6589383741602666, 'random_strength': 1.9395820094129066}. Best is trial 88 with value: 0.8872374115294746.


In [8]:
best_params = study.best_params
best_value = study.best_value

In [9]:
print(best_params)
print(best_value)

{'iterations': 900, 'learning_rate': 0.03422600888232256, 'depth': 5, 'loss_function': 'RMSE', 'subsample': 1.0, 'l2_leaf_reg': 2.7759756882947846, 'random_strength': 1.0123228282165018}
0.8872374115294746


### Évaluation finale sur le jeu de test

In [10]:
# Pipeline avec les meilleurs hyperparamètres trouvés par Optuna,
# entraîné sur le jeu d'entraînement puis évalué une seule fois sur le jeu de test
best_pipe = Pipeline([
     ('preprocessor', preprocessor),
     ('model', CatBoostRegressor(**best_params, random_state=42, verbose=False))
])
best_pipe.fit(X_train, y_train)
y_pred_test = best_pipe.predict(X_test)

print("TEST MSE :", mean_squared_error(y_test, y_pred_test))
print("TEST MAE :", mean_absolute_error(y_test, y_pred_test))
print("TEST R2 :", r2_score(y_test, y_pred_test))

TEST MSE : 9.624154920625475
TEST MAE : 1.8675503111549872
TEST R2 : 0.8687623288221056
